# Credit Card Fraud Analytics — Anomaly Detection

## Phase 5: Anomaly Detection

This phase adds an unsupervised analytics component to the project.

Objectives:
- Detect transactions that look unusual without using the fraud label during model fitting
- Use Isolation Forest as the main anomaly detection method
- Compare anomaly scores with the known fraud labels after scoring
- Evaluate how concentrated fraud is among the highest-risk anomaly scores
- Support investigation and prioritization use cases

Important:
- The `Class` label is not used to fit the anomaly detector.
- V1–V28 are anonymized PCA components.
- The dataset does not contain customer, merchant, location, or category identifiers.
- Anomaly detection identifies unusual observations; it does not prove that an observation is fraudulent.


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


## 2. Load Clean Dataset


In [ ]:
DATA_PATH = "../data/creditcard_clean.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Fraud transactions: {(df['Class'] == 1).sum():,}")
print(f"Normal transactions: {(df['Class'] == 0).sum():,}")


## 3. Define Anomaly Detection Features


In [ ]:
v_features = [f"V{i}" for i in range(1, 29)]

# Amount is included because it represents transaction magnitude.
# Time is excluded from the base model to avoid letting elapsed time dominate the anomaly structure.
feature_columns = v_features + ["Amount"]

X = df[feature_columns].copy()
y = df["Class"].copy()

print(f"Features used: {len(feature_columns)}")
print(feature_columns)


## 4. Prepare Features

The scaler is fitted only on the feature matrix and the fraud label is kept completely separate from model fitting.

Standardization is important because `Amount` is on a different scale from the PCA-derived features.


In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(f"Scaled matrix shape: {X_scaled.shape}")


## 5. Fit Isolation Forest


In [ ]:
# contamination is set close to the observed fraud prevalence.
# This is an operating-point choice for this portfolio analysis, not a claim that
# every anomaly is fraud.

fraud_rate = y.mean()

iso_forest = IsolationForest(
    n_estimators=300,
    contamination=fraud_rate,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_scaled)

print("Isolation Forest fitted successfully.")


## 6. Generate Anomaly Scores


In [ ]:
# decision_function: larger values are more normal.
# We reverse the sign so that larger values represent greater anomaly risk.

df["anomaly_score"] = -iso_forest.decision_function(X_scaled)

df["anomaly_flag"] = (
    iso_forest.predict(X_scaled) == -1
).astype(int)

display(
    df[["Amount", "Class", "anomaly_score", "anomaly_flag"]].head()
)


## 7. Anomaly Score Distribution


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df["anomaly_score"], bins=100)
plt.title("Isolation Forest Anomaly Score Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()


## 8. Anomaly Score by Class


In [ ]:
score_summary = (
    df.groupby("Class")["anomaly_score"]
    .agg(["count", "mean", "median", "min", "max"])
)

score_summary.index = ["Normal", "Fraud"]

display(score_summary)


## 9. Compare Anomaly Scores for Fraud and Normal Transactions


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(
    df.loc[df["Class"] == 0, "anomaly_score"],
    bins=100,
    alpha=0.6,
    label="Normal"
)
plt.hist(
    df.loc[df["Class"] == 1, "anomaly_score"],
    bins=50,
    alpha=0.6,
    label="Fraud"
)
plt.title("Anomaly Score Distribution: Normal vs Fraud")
plt.xlabel("Anomaly Score")
plt.ylabel("Number of Transactions")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Rank Transactions by Anomaly Score


In [ ]:
ranked_transactions = df.sort_values(
    "anomaly_score",
    ascending=False
).reset_index()

display(
    ranked_transactions[
        ["index", "Time", "Amount", "Class", "anomaly_score", "anomaly_flag"]
    ].head(20)
)


The highest anomaly scores represent the observations that the model considers most unusual.


## 11. Anomaly Detection Performance


In [ ]:
y_true = y
y_score = df["anomaly_score"]
y_pred = df["anomaly_flag"]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_score)
pr_auc = average_precision_score(y_true, y_score)

anomaly_metrics = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Value": [
        precision,
        recall,
        f1,
        roc_auc,
        pr_auc
    ]
})

display(anomaly_metrics)


### Interpretation

Precision measures how many transactions flagged as anomalies are actually fraud.

Recall measures how many known fraud transactions are captured among the anomalies.

F1 summarizes precision and recall.

ROC-AUC evaluates ranking quality across thresholds.

PR-AUC is especially informative for this highly imbalanced fraud dataset.


## 12. Confusion Matrix at the Selected Operating Point


In [ ]:
cm = confusion_matrix(y_true, y_pred)

confusion_df = pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Fraud"],
    columns=["Predicted Normal", "Predicted Fraud"]
)

display(confusion_df)


## 13. Fraud Capture by Top-N Anomalies


In [ ]:
ranked = df.sort_values(
    "anomaly_score",
    ascending=False
).reset_index(drop=True)

fraud_total = ranked["Class"].sum()

top_n_values = [50, 100, 200, 500, 1000, 2000, 5000]

top_n_results = []

for n in top_n_values:
    top_n = ranked.head(n)
    fraud_found = top_n["Class"].sum()

    top_n_results.append({
        "top_n_transactions": n,
        "fraud_found": fraud_found,
        "fraud_recall_pct": fraud_found / fraud_total * 100,
        "fraud_precision_pct": fraud_found / n * 100
    })

top_n_results = pd.DataFrame(top_n_results)

display(top_n_results)


## 14. Fraud Capture Curve


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    top_n_results["top_n_transactions"],
    top_n_results["fraud_recall_pct"],
    marker="o"
)
plt.title("Fraud Recall by Number of Highest-Risk Transactions Reviewed")
plt.xlabel("Transactions Reviewed")
plt.ylabel("Fraud Recall (%)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


This curve supports a practical investigation workflow: prioritize the highest-risk transactions first when review capacity is limited.


## 15. Top 1% and Top 5% Anomaly Segments


In [ ]:
segment_results = []

for pct in [0.01, 0.05, 0.10]:
    n = max(1, int(np.ceil(len(ranked) * pct)))
    segment = ranked.head(n)
    fraud_found = segment["Class"].sum()

    segment_results.append({
        "segment": f"Top {pct * 100:.0f}%",
        "transactions_reviewed": n,
        "fraud_found": fraud_found,
        "fraud_recall_pct": fraud_found / fraud_total * 100,
        "precision_pct": fraud_found / n * 100,
        "fraud_amount": segment.loc[segment["Class"] == 1, "Amount"].sum()
    })

segment_results = pd.DataFrame(segment_results)

display(segment_results)


## 16. Fraud Amount Captured by Anomaly Ranking


In [ ]:
total_fraud_amount = df.loc[df["Class"] == 1, "Amount"].sum()

amount_capture = []

for pct in [0.01, 0.05, 0.10]:
    n = max(1, int(np.ceil(len(ranked) * pct)))
    segment = ranked.head(n)

    captured_amount = segment.loc[
        segment["Class"] == 1, "Amount"
    ].sum()

    amount_capture.append({
        "segment": f"Top {pct * 100:.0f}%",
        "transactions_reviewed": n,
        "captured_fraud_amount": captured_amount,
        "fraud_amount_capture_pct": captured_amount / total_fraud_amount * 100
    })

amount_capture = pd.DataFrame(amount_capture)

display(amount_capture)


## 17. Top Anomalies


In [ ]:
top_anomalies = (
    ranked[
        ["Time", "Amount", "Class", "anomaly_score", "anomaly_flag"]
    ]
    .head(50)
)

display(top_anomalies)


## 18. Anomaly Rate by Amount Band


In [ ]:
amount_bins = [-0.01, 10, 25, 50, 100, 250, 500, 1000, 2500, 5000, np.inf]
amount_labels = [
    "0–10", "10–25", "25–50", "50–100", "100–250",
    "250–500", "500–1,000", "1,000–2,500",
    "2,500–5,000", "5,000+"
]

df["AmountBand"] = pd.cut(
    df["Amount"],
    bins=amount_bins,
    labels=amount_labels
)

anomaly_by_amount = (
    df.groupby("AmountBand", observed=False)
    .agg(
        transactions=("Class", "size"),
        fraud_transactions=("Class", "sum"),
        anomaly_transactions=("anomaly_flag", "sum")
    )
)

anomaly_by_amount["anomaly_rate_pct"] = (
    anomaly_by_amount["anomaly_transactions"]
    / anomaly_by_amount["transactions"] * 100
)

anomaly_by_amount["fraud_rate_pct"] = (
    anomaly_by_amount["fraud_transactions"]
    / anomaly_by_amount["transactions"] * 100
)

display(anomaly_by_amount)


## 19. Anomaly Rate by Hour


In [ ]:
df["HourOfDay"] = (df["Time"] // 3600) % 24

anomaly_by_hour = (
    df.groupby("HourOfDay")
    .agg(
        transactions=("Class", "size"),
        fraud_transactions=("Class", "sum"),
        anomaly_transactions=("anomaly_flag", "sum")
    )
)

anomaly_by_hour["anomaly_rate_pct"] = (
    anomaly_by_hour["anomaly_transactions"]
    / anomaly_by_hour["transactions"] * 100
)

anomaly_by_hour["fraud_rate_pct"] = (
    anomaly_by_hour["fraud_transactions"]
    / anomaly_by_hour["transactions"] * 100
)

display(anomaly_by_hour)


## 20. Anomaly vs Fraud Agreement


In [ ]:
agreement_table = pd.crosstab(
    df["anomaly_flag"],
    df["Class"],
    rownames=["Anomaly Flag"],
    colnames=["Actual Class"]
)

display(agreement_table)


## 21. Fraud Rate Among Anomalies


In [ ]:
fraud_rate_among_anomalies = (
    df.loc[df["anomaly_flag"] == 1, "Class"].mean() * 100
)

fraud_rate_among_normal_scored = (
    df.loc[df["anomaly_flag"] == 0, "Class"].mean() * 100
)

anomaly_group_summary = pd.DataFrame({
    "group": ["Flagged as anomaly", "Not flagged as anomaly"],
    "transactions": [
        (df["anomaly_flag"] == 1).sum(),
        (df["anomaly_flag"] == 0).sum()
    ],
    "fraud_rate_pct": [
        fraud_rate_among_anomalies,
        fraud_rate_among_normal_scored
    ]
})

display(anomaly_group_summary)


## 22. Operational Review Simulation


In [ ]:
review_budgets = [100, 250, 500, 1000, 2500, 5000]

operational_results = []

for budget in review_budgets:
    reviewed = ranked.head(budget)
    fraud_found = reviewed["Class"].sum()

    operational_results.append({
        "review_budget": budget,
        "fraud_found": fraud_found,
        "fraud_recall_pct": fraud_found / fraud_total * 100,
        "precision_pct": fraud_found / budget * 100,
        "fraud_amount_captured": reviewed.loc[
            reviewed["Class"] == 1, "Amount"
        ].sum()
    })

operational_results = pd.DataFrame(operational_results)

display(operational_results)


This table connects anomaly ranking to a realistic investigation-capacity scenario.


## 23. Save Phase 5 Outputs


In [ ]:
OUTPUT_DIR = "../data"

anomaly_metrics.to_csv(
    f"{OUTPUT_DIR}/anomaly_metrics.csv",
    index=False
)

confusion_df.to_csv(
    f"{OUTPUT_DIR}/anomaly_confusion_matrix.csv"
)

top_n_results.to_csv(
    f"{OUTPUT_DIR}/anomaly_top_n_results.csv",
    index=False
)

segment_results.to_csv(
    f"{OUTPUT_DIR}/anomaly_segments.csv",
    index=False
)

amount_capture.to_csv(
    f"{OUTPUT_DIR}/anomaly_amount_capture.csv",
    index=False
)

anomaly_by_amount.to_csv(
    f"{OUTPUT_DIR}/anomaly_by_amount_band.csv"
)

anomaly_by_hour.to_csv(
    f"{OUTPUT_DIR}/anomaly_by_hour.csv"
)

anomaly_group_summary.to_csv(
    f"{OUTPUT_DIR}/anomaly_group_summary.csv",
    index=False
)

operational_results.to_csv(
    f"{OUTPUT_DIR}/anomaly_operational_review.csv",
    index=False
)

top_anomalies.to_csv(
    f"{OUTPUT_DIR}/top_anomalies.csv",
    index=False
)

print("Phase 5 output tables saved successfully.")


## 24. Phase 5 Conclusions

After running the notebook, document evidence-based findings for:

1. How well anomaly ranking captures known fraud
2. Precision and recall at the selected operating point
3. PR-AUC and ROC-AUC
4. Fraud capture among the top 1%, 5%, and 10% of highest-risk transactions
5. Fraud monetary exposure captured by the highest-risk segments
6. Fraud concentration among the transactions prioritized for review
7. Differences between anomaly rates and observed fraud rates across amount bands and hours
8. The operational trade-off between review capacity and fraud capture

Do not describe an anomaly as fraudulent solely because the model flagged it.

### Important analytical distinction

Anomaly detection is an **unsupervised prioritization tool**.

The `Class` label is used only after scoring to evaluate how useful the anomaly ranking is for fraud investigation.

### Next Phase

**Phase 6 — Predictive Analysis**

We will build a supervised fraud classification baseline, starting with Logistic Regression and evaluating it with Precision, Recall, F1, PR-AUC, ROC-AUC, and a confusion matrix.
